# Evaluation Metrics — Practical Examples

This notebook demonstrates the four types of evaluation metrics for AI agents:

1. **Objective / code-based metrics** — deterministic checks with ground truth
2. **LLM-as-judge metrics** — subjective quality scoring via a judge model
3. **Trajectory metrics** — tool call accuracy, step efficiency
4. **Component-level metrics** — isolating individual pipeline stages

Based on the evaluation patterns used in the experience-generation-server codebase (DeepEval + LangGraph).

---

**Learning path:** Run §2–9 after reading `1-agent_evaluation_guide.md`. \
Run §10–13 after reading `3-production_evaluation_pattern.md` and `4-establishing-evaluation-framework.md`.

| Section | Builds on |
|---------|-----------|
| §2–3    | Two Axes of Evaluation (objective + ground truth) |
| §4–5    | LLM-as-judge patterns |
| §6      | Trajectory metrics (Level 2-3 agents) |
| §7      | Error analysis loop |
| §8      | Codebase score aggregation |
| §9      | Hyperparameter sweep / improvement loop |
| §10     | RAG Triad — read `5` Pillar 2 first |
| §11     | pass^k — read `5` Agent Capabilities first |
| §12     | Multi-turn eval — read `5` Conversational Eval first |
| §13     | CI/CD gate — read `3` Langfuse → Golden Dataset first |


## 1. Setup

In [ ]:
import re
import json
from typing import Optional
from urllib.parse import urlparse
from dataclasses import dataclass, field

## 2. Objective Metric: Date Extraction Accuracy

From Andrew Ng's course — the simplest form of eval.
Agent is asked to extract a due date and format it as YYYY/MM/DD.

In [ ]:
# Gold standard dataset — manually annotated
date_test_cases = [
    {"invoice_text": "Please pay by August 20, 2025",   "expected": "2025/08/20"},
    {"invoice_text": "Due date: March 15, 2025",         "expected": "2025/03/15"},
    {"invoice_text": "Payment required before 01/12/25", "expected": "2025/01/12"},
]

def extract_date_from_llm_response(llm_response: str) -> Optional[str]:
    """Extract YYYY/MM/DD pattern from LLM output."""
    date_pattern = r'\d{4}/\d{2}/\d{2}'
    matches = re.findall(date_pattern, llm_response)
    return matches[0] if matches else None

def evaluate_date_extraction(test_cases: list, llm_responses: list) -> dict:
    num_correct = 0
    for case, response in zip(test_cases, llm_responses):
        extracted = extract_date_from_llm_response(response)
        if extracted == case["expected"]:
            num_correct += 1
        else:
            print(f"FAIL: expected {case['expected']}, got {extracted}")
            print(f"      invoice: {case['invoice_text']}")
    
    accuracy = num_correct / len(test_cases)
    print(f"\nAccuracy: {num_correct}/{len(test_cases)} = {accuracy:.1%}")
    return {"accuracy": accuracy, "num_correct": num_correct, "total": len(test_cases)}

# Simulated LLM responses
simulated_responses = [
    "The due date formatted as requested is 2025/08/20",
    "Due date: 2025/03/15",
    "2025/01/13"  # wrong — simulate failure
]

evaluate_date_extraction(date_test_cases, simulated_responses)

## 3. Objective Metric: Source Quality F1-Score

For a research agent: did it cite authoritative sources?
Uses precision, recall, and F1 against a gold standard source list.

In [ ]:
def extract_domain(url: str) -> str:
    """Extract base domain from URL."""
    try:
        parsed = urlparse(url if url.startswith("http") else f"https://{url}")
        return parsed.netloc.replace("www.", "")
    except Exception:
        return url

def calculate_source_quality_f1(
    agent_sources: list[str],
    gold_sources: list[str]
) -> dict:
    """
    F1-score measuring how well agent sources overlap with gold standard.

    Precision = fraction of agent sources that are gold-standard
    Recall    = fraction of gold-standard sources that were found
    F1        = harmonic mean of precision and recall
    """
    agent_domains = set(extract_domain(u) for u in agent_sources)
    gold_domains  = set(gold_sources)

    true_positives = len(agent_domains & gold_domains)
    precision = true_positives / len(agent_domains) if agent_domains else 0.0
    recall    = true_positives / len(gold_domains)  if gold_domains  else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0.0
    )

    print(f"Agent sources:  {agent_domains}")
    print(f"Gold sources:   {gold_domains}")
    print(f"True positives: {true_positives}")
    print(f"Precision: {precision:.2f}  Recall: {recall:.2f}  F1: {f1:.2f}")
    return {"precision": precision, "recall": recall, "f1": f1}

# Example
agent_found = [
    "https://sap.com/products/ai-core",
    "https://techcrunch.com/sap-ai",
    "https://sapinsider.org/2025/cx-trends",
    "https://randomblog.io/sap-stuff",
]
gold_standard_sources = ["sap.com", "sapinsider.org", "forbes.com"]

calculate_source_quality_f1(agent_found, gold_standard_sources)

## 4. LLM-as-Judge: Talking Points Coverage

For subjective evaluation: does the generated essay cover the gold-standard talking points?
This is the Andrew Ng course approach — count coverage using a judge LLM.

In [ ]:
def build_talking_points_judge_prompt(
    original_prompt: str,
    essay_text: str,
    gold_standard_points: list[str]
) -> str:
    """
    Build the judge prompt that asks an LLM to count how many
    gold-standard talking points appear in the essay.

    Pattern from Andrew Ng's agentic AI course.
    """
    points_str = "\n".join(f"- {p}" for p in gold_standard_points)
    return f"""Determine how many of the {len(gold_standard_points)} gold-standard \
talking points are present in the provided essay.

Original Prompt:
{original_prompt}

Essay to Evaluate:
{essay_text}

Gold Standard Talking Points:
{points_str}

Output Format:
Return a JSON object with exactly two keys:
- "score": a single integer between 0 and {len(gold_standard_points)}
- "explanation": a string listing which talking points are present
"""

# Example usage (without live LLM call)
prompt = build_talking_points_judge_prompt(
    original_prompt="Recent developments in black hole science",
    essay_text="""
    The Event Horizon Telescope recently produced groundbreaking images.
    Einstein's general relativity continues to be validated by observations.
    Radio telescope arrays provide unprecedented resolution.
    """,
    gold_standard_points=[
        "Event horizon",
        "Radio telescope",
        "Einstein black hole theories",
        "Gravitational waves",
        "Hawking radiation"
    ]
)
print(prompt)

## 5. LLM-as-Judge: Rubric-Based Scoring

The pattern used in this codebase's `EvaluationWorkflow`.
Each dimension returns a score 1-5 with reasoning.

In [ ]:
# The 9 evaluation dimensions used by the codebase's EvaluationWorkflow
# See: orchestration/workflows/evaluation_workflow.py

EVALUATION_DIMENSIONS = {
    # Single-score evaluators (1-5)
    "completeness": {
        "description": "Does the report cover all topics in the outline and answer the user question?",
        "canonical_name": "report.completeness",
    },
    "structure": {
        "description": "Is the report logically structured with clear sections and good flow?",
        "canonical_name": "report.structure",
    },
    "relevance": {
        "description": "Does the content directly address the user's research question?",
        "canonical_name": "report.relevance",
    },
    # Overall quality sub-dimensions (each 1-5, averaged)
    "research_depth": {
        "description": "Depth and thoroughness of the research conducted",
        "canonical_name": "report.overall_quality.research_depth",
    },
    "source_quality": {
        "description": "Authority and reliability of sources used",
        "canonical_name": "report.overall_quality.source_quality",
    },
    "analytical_rigor": {
        "description": "Quality of analysis and reasoning applied to findings",
        "canonical_name": "report.overall_quality.analytical_rigor",
    },
    "practical_value": {
        "description": "Actionability and usefulness of the insights for the reader",
        "canonical_name": "report.overall_quality.practical_value",
    },
    "balance_and_objectivity": {
        "description": "Balanced presentation without bias or unsupported conclusions",
        "canonical_name": "report.overall_quality.balance_and_objectivity",
    },
    "writing_quality": {
        "description": "Clarity, coherence, and professional quality of the writing",
        "canonical_name": "report.overall_quality.writing_quality",
    },
}

def build_rubric_eval_prompt(dimension: str, report: str, user_question: str) -> str:
    info = EVALUATION_DIMENSIONS[dimension]
    return f"""Evaluate the following report on the dimension: {dimension}

Criterion: {info['description']}

User Question: {user_question}

Report:
{report}

Return a JSON object with:
- "score": integer from 1 (very poor) to 5 (excellent)
- "reasoning": one sentence explaining the score
"""

# Demo
print(build_rubric_eval_prompt(
    dimension="source_quality",
    report="According to sap.com and sapinsider.org, AI Core provides...",
    user_question="What are the latest SAP AI capabilities?"
))

## 6. Trajectory Metric: Tool Call Accuracy

Measures whether the agent selected the correct tool and passed correct parameters.
Critical for Level 2-3 agents.

In [ ]:
@dataclass
class ToolCall:
    tool_name: str
    arguments: dict

@dataclass 
class TrajectoryTestCase:
    prompt: str
    expected_tools: list[str]          # tools that should have been called
    actual_tool_calls: list[ToolCall]  # what the agent actually called

def evaluate_trajectory(test_case: TrajectoryTestCase) -> dict:
    """
    Compute step precision and recall for tool selection.

    Step Precision = fraction of called tools that were necessary
    Step Recall    = fraction of necessary tools that were called
    Path Efficiency = ratio of optimal steps to actual steps taken
    """
    called_tools = [tc.tool_name for tc in test_case.actual_tool_calls]
    expected_set = set(test_case.expected_tools)
    called_set   = set(called_tools)

    true_positives = len(expected_set & called_set)
    precision  = true_positives / len(called_set)   if called_set   else 0.0
    recall     = true_positives / len(expected_set) if expected_set else 0.0
    efficiency = len(expected_set) / len(called_tools) if called_tools else 0.0

    result = {
        "step_precision":  round(precision, 2),
        "step_recall":     round(recall, 2),
        "path_efficiency": round(min(efficiency, 1.0), 2),
        "unnecessary_tools": list(called_set - expected_set),
        "missed_tools":      list(expected_set - called_set),
    }
    return result

# Example: research agent should call web_search then synthesize, but also called an unnecessary tool
test_case = TrajectoryTestCase(
    prompt="What are SAP's latest AI capabilities?",
    expected_tools=["web_search", "synthesize_research"],
    actual_tool_calls=[
        ToolCall("web_search", {"query": "SAP AI capabilities 2025"}),
        ToolCall("calculator", {}),         # unnecessary
        ToolCall("synthesize_research", {}),
    ]
)

result = evaluate_trajectory(test_case)
print(json.dumps(result, indent=2))

## 7. Component Error Rate Tracking

The 'counting up errors' method from the Andrew Ng course.
Track error rates per pipeline stage across your eval dataset.

In [ ]:
@dataclass
class EvalRecord:
    prompt: str
    search_terms_ok: bool
    search_results_ok: bool
    source_selection_ok: bool
    synthesis_ok: bool
    writing_ok: bool
    failure_notes: str = ""

eval_dataset = [
    EvalRecord("Recent developments in black hole science",
               True, False, True, True, True,
               "Search returned too many blog posts, not enough papers"),
    EvalRecord("Renting vs buying a home in Seattle",
               True, True, False, True, True,
               "Missed well-known real-estate blog"),
    EvalRecord("Robotics for harvesting fruit",
               False, False, True, True, True,
               "Terms too generic; returned elementary school site"),
    EvalRecord("Batteries for electric vehicles",
               True, False, False, True, True,
               "Only selected US-based companies; missed key magazine"),
    EvalRecord("SAP CX AI capabilities 2025",
               True, True, True, True, True),
]

def compute_component_error_rates(records: list[EvalRecord]) -> dict:
    n = len(records)
    components = [
        "search_terms_ok",
        "search_results_ok",
        "source_selection_ok",
        "synthesis_ok",
        "writing_ok",
    ]
    error_rates = {}
    for comp in components:
        failures = sum(1 for r in records if not getattr(r, comp))
        error_rates[comp.replace("_ok", "")] = {
            "error_rate": failures / n,
            "failures": failures,
            "total": n
        }
    return dict(sorted(error_rates.items(), key=lambda x: -x[1]["error_rate"]))

rates = compute_component_error_rates(eval_dataset)
print(f"{'Component':<22}  {'Error Rate':>10}  {'Failures':>8}")
print("-" * 46)
for comp, stats in rates.items():
    bar = "#" * int(stats["error_rate"] * 20)
    print(f"{comp:<22}  {stats['error_rate']:>9.0%}  {stats['failures']:>4}/{stats['total']}  {bar}")
print()
top_component = list(rates.keys())[0]
print(f"=> Fix '{top_component}' first — it has the highest error rate.")

## 8. Overall Score Aggregation

Pattern used by the codebase's `evaluation_runner.py`:
collect per-metric scores → average into overall → write to observability platform.

In [ ]:
def aggregate_evaluation_scores(raw_evaluations: list[dict]) -> dict:
    """
    Aggregate per-metric evaluation results into summary and overall scores.

    Mirrors the logic in:
      orchestration/evaluation/evaluation_runner.py :: run_report_evaluation_job()
    """
    summary_scores = {}
    detailed_scores = {}

    for evaluation in raw_evaluations:
        name  = evaluation.get("name", "unknown")
        score = evaluation.get("score")
        if not isinstance(score, (int, float)):
            continue
        reasoning = evaluation.get("reasoning", "")
        summary_scores[name] = float(score)
        detailed_scores[name] = {"score": float(score), "reasoning": reasoning}

    overall = (
        sum(summary_scores.values()) / len(summary_scores)
        if summary_scores else None
    )
    if overall is not None:
        summary_scores["report.overall"] = overall
        detailed_scores["report.overall"] = {
            "score": overall,
            "reasoning": "Average of all report-level metrics."
        }

    return {"summary": summary_scores, "detailed": detailed_scores}

# Simulate output from EvaluationWorkflow
workflow_output = [
    {"name": "completeness",  "score": 4, "reasoning": "Covers most outline topics"},
    {"name": "structure",     "score": 5, "reasoning": "Clear sections, good flow"},
    {"name": "relevance",     "score": 4, "reasoning": "Directly addresses question"},
    {"name": "overall_quality_research_depth",       "score": 3, "reasoning": "Limited primary sources"},
    {"name": "overall_quality_source_quality",       "score": 4, "reasoning": "Good mix of authoritative sources"},
    {"name": "overall_quality_analytical_rigor",     "score": 3, "reasoning": "Analysis is surface-level in places"},
    {"name": "overall_quality_practical_value",      "score": 4, "reasoning": "Actionable insights provided"},
    {"name": "overall_quality_balance_and_objectivity", "score": 5, "reasoning": "Well-balanced perspective"},
    {"name": "overall_quality_writing_quality",      "score": 4, "reasoning": "Clear, professional prose"},
]

result = aggregate_evaluation_scores(workflow_output)

print("Summary scores:")
for name, score in result["summary"].items():
    bar = "*" * int(score)
    print(f"  {name:<48}  {score:.2f}  {bar}")

## 9. Hyperparameter Sweep: Tracking Metrics Across Configurations

From the Andrew Ng course: vary hyperparameters (search engine, result count, date range)
and track how each affects component scores. Find the best config empirically.

In [ ]:
@dataclass
class SearchConfig:
    search_engine: str
    num_results: int
    date_range_days: int

@dataclass
class ConfigEvalResult:
    config: SearchConfig
    source_quality_f1: float
    synthesis_coverage: float
    avg_latency_ms: float
    cost_per_run_usd: float

# Simulated sweep results (in practice, run your eval dataset against each config)
sweep_results = [
    ConfigEvalResult(SearchConfig("google",  5, 30),  0.62, 0.71, 4200, 0.12),
    ConfigEvalResult(SearchConfig("google", 10, 30),  0.74, 0.83, 6800, 0.18),
    ConfigEvalResult(SearchConfig("google", 10, 90),  0.78, 0.85, 7100, 0.19),
    ConfigEvalResult(SearchConfig("bing",   10, 30),  0.65, 0.75, 5200, 0.14),
    ConfigEvalResult(SearchConfig("bing",   10, 90),  0.69, 0.79, 5400, 0.15),
]

# Print comparison table
print(f"{'Engine':<8} {'Results':>7} {'Days':>5} | {'F1':>6} {'Coverage':>9} {'Latency(ms)':>12} {'Cost($)':>8}")
print("-" * 60)
for r in sorted(sweep_results, key=lambda x: -x.source_quality_f1):
    c = r.config
    print(f"{c.search_engine:<8} {c.num_results:>7} {c.date_range_days:>5} | "
          f"{r.source_quality_f1:>6.2f} {r.synthesis_coverage:>9.2f} "
          f"{r.avg_latency_ms:>12.0f} {r.cost_per_run_usd:>8.2f}")

best = max(sweep_results, key=lambda x: x.source_quality_f1)
print(f"\nBest config: {best.config.search_engine}, {best.config.num_results} results, {best.config.date_range_days}d window")
print(f"F1: {best.source_quality_f1:.2f}, Coverage: {best.synthesis_coverage:.2f}, Cost: ${best.cost_per_run_usd:.2f}/run")

## 10. RAG Metrics: Faithfulness & Answer Relevancy

For research agents that retrieve context before generating answers.
These metrics implement the RAG Triad described in `4-establishing-evaluation-framework.md`.

Three relationships to test independently:
- **Faithfulness** (groundedness) — is every claim in the answer supported by retrieved context?
- **Answer Relevancy** — does the answer actually address what the user asked?
- **Context Relevance** — were the retrieved chunks relevant to the query?

In [ ]:
# RAG evaluation without a live API call — pure code patterns
# In production, replace simulate_judge() with an actual LLM call.

def simulate_judge(prompt: str) -> str:
    """Stub — replace with: anthropic_client.messages.create(...) or similar."""
    # Returns a deterministic fake response for demonstration
    if "supported" in prompt.lower() or "grounded" in prompt.lower():
        return '{"answer": "Yes"}'
    return '{"answer": "No"}'


# ── Faithfulness (QAG pattern) ────────────────────────────────────────────────
# Decompose the answer into individual claims, then verify each claim
# against the retrieved context with a yes/no question.
# Score = supported_claims / total_claims
# More reliable than asking "Is this answer faithful overall?" because it
# forces the judge to reason at the claim level, not the paragraph level.

def extract_claims(answer: str) -> list[str]:
    """
    In production: call an LLM to extract atomic factual claims.
    Here we split on sentences as a stand-in.
    """
    sentences = [s.strip() for s in answer.split(".") if s.strip()]
    return sentences

def check_claim_against_context(claim: str, context_chunks: list[str]) -> bool:
    """Ask the judge LLM whether the context supports this specific claim."""
    context_text = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(context_chunks))
    prompt = f"""Does the following context support the claim? Answer only Yes or No.

Claim: {claim}

Context:
{context_text}

Return JSON: {{"answer": "Yes"}} or {{"answer": "No"}}"""
    raw = simulate_judge(prompt)
    try:
        return json.loads(raw).get("answer", "No").strip().lower() == "yes"
    except json.JSONDecodeError:
        return False

def compute_faithfulness(answer: str, context_chunks: list[str]) -> dict:
    claims = extract_claims(answer)
    if not claims:
        return {"faithfulness": 0.0, "supported": 0, "total": 0, "claims": []}

    results = []
    for claim in claims:
        supported = check_claim_against_context(claim, context_chunks)
        results.append({"claim": claim, "supported": supported})

    supported_count = sum(1 for r in results if r["supported"])
    score = supported_count / len(claims)
    return {
        "faithfulness": round(score, 3),
        "supported": supported_count,
        "total": len(claims),
        "claims": results,
    }


# ── Answer Relevancy ──────────────────────────────────────────────────────────
# Measures whether the answer directly addresses what the user asked.
# Uses cosine similarity between the question embedding and back-generated
# questions from the answer. Here we use a lightweight keyword-overlap proxy.

def compute_answer_relevancy_simple(question: str, answer: str) -> dict:
    """
    Lightweight keyword-overlap proxy for answer relevancy.
    In production use: sentence-transformers cosine similarity, or
    DeepEval AnswerRelevancyMetric, or RAGAS answer_relevancy.
    """
    q_tokens = set(question.lower().split())
    a_tokens = set(answer.lower().split())
    # Remove stopwords inline
    stopwords = {"the", "a", "an", "is", "are", "was", "were", "of", "to", "in",
                 "for", "and", "or", "it", "its", "be", "by", "on", "at", "with"}
    q_keywords = q_tokens - stopwords
    a_keywords = a_tokens - stopwords

    if not q_keywords:
        return {"answer_relevancy": 0.0}

    overlap = len(q_keywords & a_keywords)
    score = overlap / len(q_keywords)
    return {
        "answer_relevancy": round(min(score, 1.0), 3),
        "q_keywords": sorted(q_keywords),
        "matched": sorted(q_keywords & a_keywords),
    }


# ── Context Relevance ─────────────────────────────────────────────────────────
# Fraction of retrieved chunks that actually contain content relevant to the query.

def compute_context_relevance(query: str, chunks: list[str]) -> dict:
    """
    Ask the judge whether each chunk is relevant to the query.
    Score = relevant_chunks / total_chunks
    Target: >= 0.7
    """
    if not chunks:
        return {"context_relevance": 0.0, "relevant": 0, "total": 0}

    relevant_count = 0
    chunk_results = []
    for chunk in chunks:
        prompt = f"""Is the following context chunk relevant to answering this query?
Query: {query}
Chunk: {chunk}
Return JSON: {{"answer": "Yes"}} or {{"answer": "No"}}"""
        raw = simulate_judge(prompt)
        try:
            is_relevant = json.loads(raw).get("answer", "No").strip().lower() == "yes"
        except json.JSONDecodeError:
            is_relevant = False
        if is_relevant:
            relevant_count += 1
        chunk_results.append({"chunk": chunk[:80] + "...", "relevant": is_relevant})

    score = relevant_count / len(chunks)
    return {
        "context_relevance": round(score, 3),
        "relevant": relevant_count,
        "total": len(chunks),
        "chunks": chunk_results,
    }


# ── Full RAG Triad Evaluation ─────────────────────────────────────────────────

def evaluate_rag_triad(
    query: str,
    answer: str,
    retrieved_chunks: list[str],
) -> dict:
    faithfulness   = compute_faithfulness(answer, retrieved_chunks)
    relevancy      = compute_answer_relevancy_simple(query, answer)
    ctx_relevance  = compute_context_relevance(query, retrieved_chunks)

    triad = {
        "faithfulness":      faithfulness["faithfulness"],
        "answer_relevancy":  relevancy["answer_relevancy"],
        "context_relevance": ctx_relevance["context_relevance"],
    }
    triad["rag_triad_avg"] = round(sum(triad.values()) / 3, 3)

    # Diagnosis
    if triad["context_relevance"] < 0.7:
        diagnosis = "RETRIEVER issue — wrong chunks fetched. Fix: embedding model, chunk size, top-k."
    elif triad["faithfulness"] < 0.8:
        diagnosis = "GENERATOR issue — hallucinating despite good context. Fix: prompt constraints, temperature."
    elif triad["answer_relevancy"] < 0.8:
        diagnosis = "ALIGNMENT issue — answer is factual but off-topic. Fix: system prompt, query reformulation."
    else:
        diagnosis = "System working correctly."

    return {**triad, "diagnosis": diagnosis, "_detail": {
        "faithfulness_detail": faithfulness,
        "relevancy_detail":    relevancy,
        "ctx_relevance_detail": ctx_relevance,
    }}


# ── Example ───────────────────────────────────────────────────────────────────

query = "What is SAP AI Core and how is it priced?"

retrieved_chunks = [
    "SAP AI Core is a managed AI service on SAP BTP that enables deployment and execution of AI functions.",
    "SAP AI Core offers three service plans: Free, Standard, and Extended.",
    "The BTP platform supports multiple runtimes including Cloud Foundry and Kyma.",  # less relevant
]

answer = (
    "SAP AI Core is a managed AI service on SAP BTP. "
    "It provides three pricing tiers: Free, Standard, and Extended. "
    "It also supports quantum computing."  # hallucinated claim
)

result = evaluate_rag_triad(query, answer, retrieved_chunks)

print("RAG Triad Scores")
print("─" * 40)
for k, v in result.items():
    if k != "_detail":
        print(f"  {k:<25} {v}")
print()
print("Faithfulness claim breakdown:")
for c in result["_detail"]["faithfulness_detail"]["claims"]:
    status = "✓" if c["supported"] else "✗"
    print(f"  {status}  {c['claim']}")

## 11. Consistency / pass^k Metric

A single successful run is not enough for mission-critical agents.
`pass^k` measures whether the agent succeeds on **all** k independent runs of the
same test case — not just the average.

- `success_rate` (average) is acceptable for standard workflows (≥ 0.9)
- `pass_at_k` (all k must pass) is required for mission-critical workflows (k ≥ 3)

Use this when an agent controls irreversible actions (send email, modify records,
trigger payments) where a single wrong run is not recoverable.

In [ ]:
import random

@dataclass
class AgentTestCase:
    input: str
    expected_output: str  # or a callable that checks correctness

def simulate_agent_run(test_input: str, noise: float = 0.15) -> str:
    """
    Stub agent that occasionally fails — simulates non-deterministic behaviour.
    Replace with your actual agent call in production.
    noise: probability of returning a wrong answer on any single run.
    """
    if random.random() < noise:
        return "WRONG_ANSWER"
    return f"Correct answer for: {test_input}"

def evaluate_output(actual: str, expected: str) -> bool:
    """Return True if the agent's output matches what was expected."""
    return actual == expected or actual.startswith("Correct")

def pass_at_k(
    test_case: AgentTestCase,
    k: int = 5,
    noise: float = 0.15,
    seed: int = 42,
) -> dict:
    """
    Run the agent k times on the same input.

    pass_at_1:    did the first run succeed?              (weakest signal)
    success_rate: what fraction of k runs succeeded?      (average — for standard workflows)
    pass_at_k:    did ALL k runs succeed?                 (strictest — for mission-critical)

    For production-critical workflows: require pass_at_k = True with k >= 3.
    For standard workflows:           success_rate >= 0.9 is acceptable.
    """
    random.seed(seed)
    results = []
    for i in range(k):
        output  = simulate_agent_run(test_case.input, noise=noise)
        passed  = evaluate_output(output, test_case.expected_output)
        results.append({"run": i + 1, "output": output, "passed": passed})

    successes = [r["passed"] for r in results]
    return {
        "pass_at_1":    successes[0],
        "pass_at_k":    all(successes),        # True only if ALL k pass
        "success_rate": round(sum(successes) / k, 3),
        "k":            k,
        "runs":         results,
    }


# ── Example 1: standard workflow — success_rate threshold ────────────────────
stable_case = AgentTestCase(
    input="What is SAP AI Core?",
    expected_output="SAP AI Core is a managed AI service."
)
result_stable = pass_at_k(stable_case, k=5, noise=0.05)  # 5% failure rate
print("Standard workflow (5% failure rate, k=5):")
print(f"  pass_at_1:    {result_stable['pass_at_1']}")
print(f"  success_rate: {result_stable['success_rate']}")
print(f"  pass_at_k:    {result_stable['pass_at_k']}  ← ALL 5 must pass\n")

# ── Example 2: flaky workflow — catches what success_rate misses ─────────────
flaky_case = AgentTestCase(
    input="Summarise and send the weekly report",  # irreversible action
    expected_output="Report sent."
)
result_flaky = pass_at_k(flaky_case, k=5, noise=0.25)  # 25% failure rate
print("Mission-critical workflow (25% failure rate, k=5):")
print(f"  pass_at_1:    {result_flaky['pass_at_1']}")
print(f"  success_rate: {result_flaky['success_rate']}  ← looks acceptable?")
print(f"  pass_at_k:    {result_flaky['pass_at_k']}  ← reveals the real problem")
print()
for r in result_flaky["runs"]:
    status = "PASS" if r["passed"] else "FAIL"
    print(f"  Run {r['run']}: {status}  →  {r['output'][:50]}")

print()
print("Decision rule:")
print("  Mission-critical (irreversible actions): require pass_at_k=True, k>=3")
print("  Standard workflows:                      require success_rate>=0.9")

## 12. Multi-Turn / Conversational Evaluation

Single-turn metrics evaluate one exchange in isolation.
Multi-turn metrics check whether the agent maintains context, memory, and coherence
across an entire conversation.

Key failure modes that single-turn metrics miss:
- **Context drift** — agent forgets a fact stated 3 turns ago
- **Pronoun resolution failure** — "it" or "that" refers to a prior turn the agent has forgotten
- **Role drift** — agent gradually abandons its defined persona across many turns
- **Conversation completeness** — all user goals addressed by the end, not just the last turn

In [ ]:
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class ConversationTurn:
    user_input:     str
    agent_response: str
    # Optional: retrieved context used for this turn (for per-turn faithfulness)
    context_chunks: list[str] = field(default_factory=list)

@dataclass
class ConversationTestCase:
    turns:          list[ConversationTurn]
    injected_facts: dict[str, str] = field(default_factory=dict)
    # injected_facts: facts explicitly stated by user that agent must later recall
    # e.g. {"company_name": "Acme Corp"}  → injected in turn 1, queried in turn 5


# ── Metric 1: Knowledge Retention ────────────────────────────────────────────
# Tests whether facts stated early in the conversation are recalled correctly later.
# Failure = context degradation. Most agents degrade after 20-50 turns.

def evaluate_knowledge_retention(
    conversation: ConversationTestCase,
    facts_to_check: dict[str, str],   # fact_key → expected value
) -> dict:
    """
    Check whether the agent's responses across the conversation contain
    the expected recalled facts.

    In production: use an LLM judge per fact. Here we use substring matching.
    """
    results = {}
    full_transcript = " ".join(t.agent_response for t in conversation.turns)

    for fact_key, expected_value in facts_to_check.items():
        recalled = expected_value.lower() in full_transcript.lower()
        results[fact_key] = {
            "expected": expected_value,
            "recalled": recalled,
        }

    retention_score = sum(1 for r in results.values() if r["recalled"]) / len(results)
    return {
        "retention_score": round(retention_score, 3),
        "facts": results,
    }


# ── Metric 2: Turn Relevancy ──────────────────────────────────────────────────
# Each agent response should address the user's input in context of prior turns.
# Uses keyword overlap as a proxy; replace with LLM judge in production.

def evaluate_turn_relevancy(conversation: ConversationTestCase) -> dict:
    stopwords = {"the", "a", "an", "is", "it", "to", "of", "in", "and", "or",
                 "i", "my", "me", "you", "your", "what", "how", "was", "for"}
    turn_scores = []

    for i, turn in enumerate(conversation.turns):
        q_words = set(turn.user_input.lower().split()) - stopwords
        a_words = set(turn.agent_response.lower().split()) - stopwords
        if not q_words:
            turn_scores.append(1.0)
            continue
        overlap = len(q_words & a_words) / len(q_words)
        turn_scores.append(round(min(overlap, 1.0), 3))

    return {
        "avg_turn_relevancy": round(sum(turn_scores) / len(turn_scores), 3),
        "per_turn":           [{"turn": i + 1, "score": s} for i, s in enumerate(turn_scores)],
        "min_turn_score":     min(turn_scores),
        "worst_turn":         turn_scores.index(min(turn_scores)) + 1,
    }


# ── Metric 3: Conversation Completeness ───────────────────────────────────────
# Were all stated user goals addressed somewhere in the conversation?

def evaluate_conversation_completeness(
    conversation: ConversationTestCase,
    user_goals: list[str],            # list of goals the user stated
) -> dict:
    """
    Check whether each user goal was addressed in the full conversation.
    In production: use an LLM judge per goal.
    """
    full_transcript = " ".join(t.agent_response for t in conversation.turns)
    results = []
    for goal in user_goals:
        # Proxy: check if key terms from the goal appear in any response
        goal_words = set(goal.lower().split()) - {"what", "how", "the", "a", "is"}
        response_words = set(full_transcript.lower().split())
        addressed = len(goal_words & response_words) / len(goal_words) >= 0.5 if goal_words else False
        results.append({"goal": goal, "addressed": addressed})

    completeness = sum(1 for r in results if r["addressed"]) / len(results)
    return {
        "completeness_score": round(completeness, 3),
        "goals":              results,
    }


# ── Example conversation ───────────────────────────────────────────────────────

conversation = ConversationTestCase(
    turns=[
        ConversationTurn(
            user_input="My company is Acme Corp and we are evaluating SAP AI Core.",
            agent_response="Welcome! Happy to help Acme Corp evaluate SAP AI Core. "
                           "SAP AI Core is a managed AI service on SAP BTP. What would you like to know?"
        ),
        ConversationTurn(
            user_input="What pricing tiers does it have?",
            agent_response="SAP AI Core offers three tiers: Free, Standard, and Extended. "
                           "The Free tier is good for prototyping."
        ),
        ConversationTurn(
            user_input="Compare that to Azure OpenAI pricing.",
            agent_response="Compared to Azure OpenAI, SAP AI Core Standard tier is typically "
                           "more cost-effective for SAP BTP workloads due to integrated billing."
        ),
        ConversationTurn(
            user_input="Which tier would you recommend for us?",
            agent_response="For Acme Corp, I would recommend the Standard tier initially. "
                           "It covers production workloads and you can upgrade to Extended as usage grows."
        ),
    ],
    injected_facts={"company_name": "Acme Corp"},
)

user_goals = [
    "understand SAP AI Core pricing",
    "compare SAP AI Core to Azure OpenAI",
    "get a tier recommendation",
]

# ── Run all three metrics ──────────────────────────────────────────────────────

retention = evaluate_knowledge_retention(
    conversation,
    facts_to_check=conversation.injected_facts,
)
relevancy    = evaluate_turn_relevancy(conversation)
completeness = evaluate_conversation_completeness(conversation, user_goals)

print("Multi-Turn Evaluation Results")
print("=" * 50)

print(f"\nKnowledge Retention:  {retention['retention_score']:.0%}")
for fact, r in retention["facts"].items():
    status = "✓ recalled" if r["recalled"] else "✗ forgotten"
    print(f"  {fact}: '{r['expected']}'  →  {status}")

print(f"\nTurn Relevancy:  avg={relevancy['avg_turn_relevancy']:.2f}  "
      f"worst=turn {relevancy['worst_turn']} ({relevancy['min_turn_score']:.2f})")
for t in relevancy["per_turn"]:
    bar = "█" * int(t["score"] * 10)
    print(f"  Turn {t['turn']}: {t['score']:.2f}  {bar}")

print(f"\nConversation Completeness:  {completeness['completeness_score']:.0%}")
for g in completeness["goals"]:
    status = "✓" if g["addressed"] else "✗"
    print(f"  {status}  {g['goal']}")

print()
print("Decision thresholds (recommended):")
print("  Knowledge retention  ≥ 0.9  (agent recalls injected facts)")
print("  Avg turn relevancy   ≥ 0.7  (each response addresses the user)")
print("  Completeness         ≥ 0.8  (all user goals addressed by end)")

## 13. CI/CD Quality Gate

Evaluation maturity Level 5 (from `1-agent_evaluation_guide.md`) means evals gate every deployment.
This section shows the concrete pattern: run the golden dataset on every PR and block merge
if `report.overall` drops more than 0.3 points versus the stored baseline.

This reuses the existing `EvaluationWorkflow`, `golden_dataset.json`, and Langfuse scores —
no new tooling required.

```
PR opened
    │
    ▼
run_ci_eval_check()
    │ loads golden_dataset.json
    │ runs EvaluationWorkflow on each record
    │ compares scores to baseline
    │
    ├── all metrics within threshold → PASS → merge allowed
    └── any metric drops > 0.3      → FAIL → merge blocked
```

In [ ]:
import json
import sys
from pathlib import Path
from dataclasses import dataclass, field

# ── Configuration ─────────────────────────────────────────────────────────────

ALERT_THRESHOLD   = 0.3    # block merge if any metric drops more than this vs baseline
REQUIRED_PASS_PCT = 0.85   # at least 85% of golden records must pass
SCORE_NAMES = [
    "report.overall",
    "report.completeness",
    "report.structure",
    "report.relevance",
    "report.overall_quality.research_depth",
    "report.overall_quality.source_quality",
]

# Baseline scores — computed from the first 50-100 prod runs; update quarterly.
# In a real CI pipeline these would live in a versioned file (e.g. baselines.json).
BASELINE_SCORES = {
    "report.overall":                            3.8,
    "report.completeness":                       3.9,
    "report.structure":                          4.1,
    "report.relevance":                          3.7,
    "report.overall_quality.research_depth":     3.5,
    "report.overall_quality.source_quality":     3.6,
}


# ── Simulate running EvaluationWorkflow on one golden record ──────────────────
# In production: replace this with an actual call to EvaluationWorkflow.ainvoke()
# or schedule_report_evaluation_job() and wait for the EvaluationRun to complete.

import random

def simulate_evaluation_workflow(record: dict, score_drift: float = 0.0) -> dict:
    """
    Stub that simulates EvaluationWorkflow scores for a golden record.
    score_drift: negative = simulate a regression (scores go down).
    """
    random.seed(hash(record["query"]) % 10000)
    scores = {}
    for name, baseline in BASELINE_SCORES.items():
        noise = random.gauss(0, 0.15)
        scores[name] = round(max(1.0, min(5.0, baseline + noise + score_drift)), 2)
    return scores


# ── Core CI/CD check function ─────────────────────────────────────────────────

@dataclass
class MetricResult:
    name:      str
    baseline:  float
    actual:    float
    delta:     float
    passed:    bool

@dataclass
class CIEvalResult:
    record_id:      str
    query:          str
    metric_results: list[MetricResult]
    record_passed:  bool   # True if ALL metrics for this record pass

def run_eval_on_record(record: dict, score_drift: float = 0.0) -> CIEvalResult:
    scores = simulate_evaluation_workflow(record, score_drift)
    metric_results = []
    for name, baseline in BASELINE_SCORES.items():
        actual  = scores.get(name, baseline)
        delta   = round(actual - baseline, 3)
        passed  = delta >= -ALERT_THRESHOLD
        metric_results.append(MetricResult(
            name=name, baseline=baseline, actual=actual, delta=delta, passed=passed
        ))
    record_passed = all(m.passed for m in metric_results)
    return CIEvalResult(
        record_id=record.get("trace_id", "unknown"),
        query=record.get("query", "")[:60],
        metric_results=metric_results,
        record_passed=record_passed,
    )


def run_ci_eval_check(
    dataset_path: str = "golden_dataset.json",
    score_drift:  float = 0.0,    # inject a simulated regression for demo
    exit_on_fail: bool = False,   # set True in real CI to fail the pipeline step
) -> dict:
    """
    Run the EvaluationWorkflow across the full golden dataset and report pass/fail.

    Returns a summary dict. In CI: sys.exit(1) if failed (controlled by exit_on_fail).

    How to wire into CI (GitHub Actions example):
        - name: Run eval quality gate
          run: python -c "from eval_ci import run_ci_eval_check; run_ci_eval_check(exit_on_fail=True)"
          env:
            LANGFUSE_PUBLIC_KEY: ${{ secrets.LANGFUSE_PUBLIC_KEY }}
            LANGFUSE_SECRET_KEY: ${{ secrets.LANGFUSE_SECRET_KEY }}
    """
    # Load golden dataset — fall back to a minimal fixture if file absent (for demo)
    if Path(dataset_path).exists():
        golden = json.loads(Path(dataset_path).read_text())
    else:
        golden = [
            {"trace_id": "fixture-001", "query": "What is SAP AI Core and how is it priced?"},
            {"trace_id": "fixture-002", "query": "Compare SAP CX AI features to competitors."},
            {"trace_id": "fixture-003", "query": "What are the latest SAP AI capabilities for 2025?"},
            {"trace_id": "fixture-004", "query": "How does SAP GenAI Hub integrate with BTP?"},
            {"trace_id": "fixture-005", "query": "What are SAP AI Core quota limits?"},
        ]

    print(f"CI/CD Eval Quality Gate")
    print(f"{'─' * 60}")
    print(f"Dataset records : {len(golden)}")
    print(f"Alert threshold : Δ > -{ALERT_THRESHOLD} per metric")
    print(f"Pass requirement: ≥ {REQUIRED_PASS_PCT:.0%} of records must pass")
    print()

    record_results = [run_eval_on_record(r, score_drift) for r in golden]

    # ── Per-record summary ────────────────────────────────────────────────────
    passed_records = sum(1 for r in record_results if r.record_passed)
    failed_records = [r for r in record_results if not r.record_passed]
    pass_pct = passed_records / len(record_results)

    # ── Metric-level aggregation ──────────────────────────────────────────────
    metric_avgs: dict[str, list[float]] = {name: [] for name in BASELINE_SCORES}
    metric_deltas: dict[str, list[float]] = {name: [] for name in BASELINE_SCORES}
    for r in record_results:
        for m in r.metric_results:
            metric_avgs[m.name].append(m.actual)
            metric_deltas[m.name].append(m.delta)

    print(f"{'Metric':<48}  {'Baseline':>8}  {'Actual':>8}  {'Δ':>7}  {'Status':>6}")
    print("─" * 85)
    all_metrics_pass = True
    for name in BASELINE_SCORES:
        avgs    = metric_avgs[name]
        avg     = sum(avgs) / len(avgs)
        delta   = avg - BASELINE_SCORES[name]
        ok      = delta >= -ALERT_THRESHOLD
        if not ok:
            all_metrics_pass = False
        flag    = "PASS" if ok else "FAIL"
        d_str   = f"{delta:+.2f}"
        print(f"  {name:<46}  {BASELINE_SCORES[name]:>8.2f}  {avg:>8.2f}  {d_str:>7}  {flag:>6}")

    # ── Overall verdict ───────────────────────────────────────────────────────
    gate_passed = all_metrics_pass and (pass_pct >= REQUIRED_PASS_PCT)

    print()
    print(f"Records passed  : {passed_records}/{len(record_results)} ({pass_pct:.0%})")
    if failed_records:
        print(f"Failed records  :")
        for r in failed_records[:3]:
            failing = [m.name.split(".")[-1] for m in r.metric_results if not m.passed]
            print(f"  {r.record_id}  failing: {', '.join(failing)}")
            print(f"    '{r.query}'")

    print()
    verdict = "PASS — safe to merge" if gate_passed else "FAIL — merge blocked"
    print(f"{'=' * 60}")
    print(f"  CI/CD VERDICT:  {verdict}")
    print(f"{'=' * 60}")

    if not gate_passed and exit_on_fail:
        sys.exit(1)   # fails the CI pipeline step

    return {
        "passed": gate_passed,
        "verdict": verdict,
        "records_passed": passed_records,
        "records_total": len(record_results),
        "pass_pct": pass_pct,
    }


# ── Demo: no regression ───────────────────────────────────────────────────────
print("Scenario A: no regression (normal run)\n")
result_ok = run_ci_eval_check(score_drift=0.0)

print("\n\n" + "=" * 60)
print("Scenario B: regression introduced (score_drift = -0.5)\n")
result_fail = run_ci_eval_check(score_drift=-0.5)